In [2]:
from pathlib import Path
import pandas as pd

TRAIN_PATH = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\data\processed\train_preprocessed.csv"
)

TEST_PATH = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\data\processed\test_preprocessed.csv"
)

# Load datasets
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)

# Ensure test IDs are treated as identifiers
if "test_id" in test_df.columns:
    test_df["test_id"] = test_df["test_id"].astype(str)

# Display dataset information
print("Training data:", train_df.shape)
print("Test data:", test_df.shape)

print("\nTraining columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\nTraining data types:\n", train_df.dtypes)
print("\nTest data types:\n", test_df.dtypes)

Training data: (404290, 6)
Test data: (3563475, 3)

Training columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']
Test columns: ['test_id', 'question1', 'question2']

Training data types:
 id               int64
qid1             int64
qid2             int64
question1       object
question2       object
is_duplicate     int64
dtype: object

Test data types:
 test_id      object
question1    object
question2    object
dtype: object


In [3]:
from sklearn.model_selection import train_test_split

TARGET = "is_duplicate"
QUESTION_COLUMNS = ["question1", "question2"]

# Separate input questions and target
X = train_df[QUESTION_COLUMNS].fillna("")
y = train_df[TARGET].astype("int8")

# Split into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Prepare test questions
X_test = test_df[QUESTION_COLUMNS].fillna("")

print("Training samples:", len(X_train))
print("Validation samples:", len(X_valid))
print("Test samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

Training samples: 323432
Validation samples: 80858
Test samples: 3563475

Training target distribution:
is_duplicate
0    0.630803
1    0.369197
Name: proportion, dtype: float64


In [4]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True).round(4))

print("\nMissing values:")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_valid.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

Training target distribution:
is_duplicate
0    0.6308
1    0.3692
Name: proportion, dtype: float64

Validation target distribution:
is_duplicate
0    0.6308
1    0.3692
Name: proportion, dtype: float64

Missing values:
Training: 0
Validation: 0
Test: 0


In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=30000,
    min_df=3,
    sublinear_tf=True,
    dtype=np.float32,
    norm="l2"
)

In [6]:
train_questions = pd.concat(
    [X_train["question1"], X_train["question2"]],
    ignore_index=True
)

word_vectorizer.fit(train_questions)

del train_questions

print("Word vocabulary size:", len(word_vectorizer.vocabulary_))

Word vocabulary size: 30000


In [7]:
def calculate_tfidf_similarity(
    df,
    vectorizer,
    batch_size=5000
):
    similarities = np.empty(len(df), dtype=np.float32)

    for start in range(0, len(df), batch_size):
        end = min(start + batch_size, len(df))

        q1 = vectorizer.transform(
            df["question1"].iloc[start:end]
        )
        q2 = vectorizer.transform(
            df["question2"].iloc[start:end]
        )

        similarities[start:end] = np.asarray(
            q1.multiply(q2).sum(axis=1)
        ).ravel()

        if start % (batch_size * 20) == 0:
            print(f"Processed {end:,} / {len(df):,} pairs")

    return similarities

In [8]:
word_sim_train = calculate_tfidf_similarity(
    X_train, word_vectorizer
)

word_sim_valid = calculate_tfidf_similarity(
    X_valid, word_vectorizer
)

print("Training features:", word_sim_train.shape)
print("Validation features:", word_sim_valid.shape)

Processed 5,000 / 323,432 pairs
Processed 105,000 / 323,432 pairs
Processed 205,000 / 323,432 pairs
Processed 305,000 / 323,432 pairs
Processed 5,000 / 80,858 pairs
Training features: (323432,)
Validation features: (80858,)


In [11]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz

# 1. Generate handcrafted features
def extract_pair_features(q1, q2):
    q1 = str(q1)
    q2 = str(q2)

    words1 = set(q1.split())
    words2 = set(q2.split())

    common_words = words1 & words2
    total_words = words1 | words2

    q1_word_count = len(q1.split())
    q2_word_count = len(q2.split())

    return {
        "q1_char_count": len(q1),
        "q2_char_count": len(q2),
        "char_count_diff": abs(len(q1) - len(q2)),
        "q1_word_count": q1_word_count,
        "q2_word_count": q2_word_count,
        "word_count_diff": abs(q1_word_count - q2_word_count),
        "char_length_ratio": (
            min(len(q1), len(q2)) / max(len(q1), len(q2))
            if max(len(q1), len(q2)) else 0
        ),
        "word_length_ratio": (
            min(q1_word_count, q2_word_count) /
            max(q1_word_count, q2_word_count)
            if max(q1_word_count, q2_word_count) else 0
        ),
        "common_word_count": len(common_words),
        "unique_word_count_q1": len(words1),
        "unique_word_count_q2": len(words2),
        "word_jaccard_similarity": (
            len(common_words) / len(total_words)
            if total_words else 0
        ),
        "common_word_ratio_q1": (
            len(common_words) / len(words1) if words1 else 0
        ),
        "common_word_ratio_q2": (
            len(common_words) / len(words2) if words2 else 0
        ),
        "fuzzy_ratio": fuzz.ratio(q1, q2),
        "token_sort_ratio": fuzz.token_sort_ratio(q1, q2),
        "token_set_ratio": fuzz.token_set_ratio(q1, q2)
    }


def create_handcrafted_features(df):
    features = [
        extract_pair_features(q1, q2)
        for q1, q2 in zip(df["question1"], df["question2"])
    ]
    return pd.DataFrame(features, index=df.index, dtype=np.float32)


# 2. Generate handcrafted features
X_train_hand = create_handcrafted_features(X_train)
X_valid_hand = create_handcrafted_features(X_valid)

print("Handcrafted training shape:", X_train_hand.shape)
print("Handcrafted validation shape:", X_valid_hand.shape)


# 3. Create TF-IDF similarity DataFrames
word_train_df = pd.DataFrame({
    "word_tfidf_similarity": word_sim_train
})

word_valid_df = pd.DataFrame({
    "word_tfidf_similarity": word_sim_valid
})


# 4. Combine all features
X_train_final = pd.concat(
    [
        X_train_hand.reset_index(drop=True),
        word_train_df.reset_index(drop=True)
    ],
    axis=1
)

X_valid_final = pd.concat(
    [
        X_valid_hand.reset_index(drop=True),
        word_valid_df.reset_index(drop=True)
    ],
    axis=1
)


# 5. Validate the final feature matrices
assert X_train_final.shape[1] == X_valid_final.shape[1]
assert X_train_final.columns.equals(X_valid_final.columns)
assert np.isfinite(X_train_final.to_numpy()).all()
assert np.isfinite(X_valid_final.to_numpy()).all()

print("\nFinal training shape:", X_train_final.shape)
print("Final validation shape:", X_valid_final.shape)
print("\nFeature preparation completed successfully.")

Handcrafted training shape: (323432, 17)
Handcrafted validation shape: (80858, 17)

Final training shape: (323432, 18)
Final validation shape: (80858, 18)

Feature preparation completed successfully.


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        C=1.0,
        random_state=42
    ))
])

In [13]:
model.fit(X_train_final, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    classification_report
)

# Predict labels and probabilities
y_valid_pred = model.predict(X_valid_final)
y_valid_prob = model.predict_proba(X_valid_final)[:, 1]

# Calculate evaluation metrics
print("Validation Results")
print("-------------------")
print("Accuracy :", accuracy_score(y_valid, y_valid_pred))
print("Precision:", precision_score(y_valid, y_valid_pred))
print("Recall   :", recall_score(y_valid, y_valid_pred))
print("F1-score :", f1_score(y_valid, y_valid_pred))
print("Log loss :", log_loss(y_valid, y_valid_prob))

print("\nClassification Report:")
print(classification_report(y_valid, y_valid_pred))

Validation Results
-------------------
Accuracy : 0.7136956145341216
Precision: 0.6153660803414684
Recall   : 0.5988342880112552
F1-score : 0.6069876409072389
Log loss : 0.5145689288804035

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.78      0.77     51005
           1       0.62      0.60      0.61     29853

    accuracy                           0.71     80858
   macro avg       0.69      0.69      0.69     80858
weighted avg       0.71      0.71      0.71     80858



In [15]:
from sklearn.base import clone
from sklearn.metrics import log_loss
import pandas as pd

C_VALUES = [0.01, 0.1, 1.0, 10.0]

results = []

for c in C_VALUES:
    candidate = clone(model)
    candidate.set_params(classifier__C=c)

    candidate.fit(X_train_final, y_train)

    probabilities = candidate.predict_proba(X_valid_final)[:, 1]
    score = log_loss(y_valid, probabilities)

    results.append({
        "C": c,
        "Validation Log Loss": score
    })

    print(f"C={c}: Log Loss={score:.6f}")

results_df = pd.DataFrame(results).sort_values(
    "Validation Log Loss"
).reset_index(drop=True)

print("\nComparison:")
print(results_df.to_string(index=False))

C=0.01: Log Loss=0.514867
C=0.1: Log Loss=0.514574
C=1.0: Log Loss=0.514569
C=10.0: Log Loss=0.514566

Comparison:
    C  Validation Log Loss
10.00             0.514566
 1.00             0.514569
 0.10             0.514574
 0.01             0.514867


In [16]:
best_c = results_df.loc[0, "C"]

print("Selected C:", best_c)
print("Lowest validation log loss:",
      results_df.loc[0, "Validation Log Loss"])

Selected C: 10.0
Lowest validation log loss: 0.5145662690682787


In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42
)

hgb_model.fit(X_train_final, y_train)

print("HistGradientBoosting training completed.")
print("Iterations used:", hgb_model.n_iter_)

c:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Program Files\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Progr

HistGradientBoosting training completed.
Iterations used: 200


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

hgb_prob = hgb_model.predict_proba(X_valid_final)[:, 1]
hgb_pred = (hgb_prob >= 0.5).astype("int8")

hgb_results = {
    "Model": "HistGradientBoosting",
    "Log Loss": log_loss(y_valid, hgb_prob),
    "Accuracy": accuracy_score(y_valid, hgb_pred),
    "Precision": precision_score(y_valid, hgb_pred),
    "Recall": recall_score(y_valid, hgb_pred),
    "F1-score": f1_score(y_valid, hgb_pred)
}

print(pd.DataFrame([hgb_results]).to_string(index=False))

               Model  Log Loss  Accuracy  Precision   Recall  F1-score
HistGradientBoosting  0.456965  0.750105   0.646883 0.711587  0.677694


In [20]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Log Loss": 0.5145662690682787,
        "Accuracy": 0.7136956,
        "F1-score": 0.6069876
    },
    {
        "Model": "HistGradientBoosting",
        "Log Loss": hgb_results["Log Loss"],
        "Accuracy": hgb_results["Accuracy"],
        "F1-score": hgb_results["F1-score"]
    }
])

print(comparison.sort_values("Log Loss").to_string(index=False))

               Model  Log Loss  Accuracy  F1-score
HistGradientBoosting  0.456965  0.750105  0.677694
 Logistic Regression  0.514566  0.713696  0.606988


In [21]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    early_stopping_rounds=50,
    n_jobs=4,
    random_state=42
)

print("XGBoost model initialized.")

XGBoost model initialized.


In [22]:
xgb_model.fit(
    X_train_final,
    y_train,
    eval_set=[(X_valid_final, y_valid)],
    verbose=50
)

print("\nXGBoost training completed.")
print("Best iteration:", xgb_model.best_iteration)
print("Best validation log loss:", xgb_model.best_score)

[0]	validation_0-logloss:0.64563
[50]	validation_0-logloss:0.48295
[100]	validation_0-logloss:0.47020
[150]	validation_0-logloss:0.46525
[200]	validation_0-logloss:0.46203
[250]	validation_0-logloss:0.45989
[300]	validation_0-logloss:0.45825
[350]	validation_0-logloss:0.45699
[400]	validation_0-logloss:0.45609
[450]	validation_0-logloss:0.45529
[500]	validation_0-logloss:0.45463
[550]	validation_0-logloss:0.45403
[600]	validation_0-logloss:0.45353
[650]	validation_0-logloss:0.45310
[700]	validation_0-logloss:0.45272
[750]	validation_0-logloss:0.45241
[800]	validation_0-logloss:0.45221
[850]	validation_0-logloss:0.45190
[900]	validation_0-logloss:0.45164
[950]	validation_0-logloss:0.45143
[999]	validation_0-logloss:0.45130

XGBoost training completed.
Best iteration: 999
Best validation log loss: 0.4513005563745242


In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    classification_report
)

xgb_prob = xgb_model.predict_proba(X_valid_final)[:, 1]
xgb_pred = (xgb_prob >= 0.5).astype("int8")

xgb_results = {
    "Model": "XGBoost",
    "Log Loss": log_loss(y_valid, xgb_prob),
    "Accuracy": accuracy_score(y_valid, xgb_pred),
    "Precision": precision_score(y_valid, xgb_pred),
    "Recall": recall_score(y_valid, xgb_pred),
    "F1-score": f1_score(y_valid, xgb_pred)
}

print("XGBoost Validation Results")
print("--------------------------")

for metric, value in xgb_results.items():
    if metric != "Model":
        print(f"{metric}: {value:.6f}")

print("\nClassification Report:")
print(classification_report(y_valid, xgb_pred))

XGBoost Validation Results
--------------------------
Log Loss: 0.451301
Accuracy: 0.754100
Precision: 0.653925
Recall: 0.709409
F1-score: 0.680538

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.78      0.80     51005
           1       0.65      0.71      0.68     29853

    accuracy                           0.75     80858
   macro avg       0.74      0.74      0.74     80858
weighted avg       0.76      0.75      0.76     80858



In [24]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Log Loss": 0.5145662691,
        "Accuracy": 0.7136956,
        "F1-score": 0.6069876
    },
    {
        "Model": "HistGradientBoosting",
        "Log Loss": 0.456965,
        "Accuracy": 0.750105,
        "F1-score": 0.677694
    },
    {
        "Model": xgb_results["Model"],
        "Log Loss": xgb_results["Log Loss"],
        "Accuracy": xgb_results["Accuracy"],
        "F1-score": xgb_results["F1-score"]
    }
])

print(comparison.to_string(index=False))

               Model  Log Loss  Accuracy  F1-score
 Logistic Regression  0.514566  0.713696  0.606988
HistGradientBoosting  0.456965  0.750105  0.677694
             XGBoost  0.451301  0.754100  0.680538


In [25]:
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import pandas as pd

param_configs = [
    {
        "max_depth": 4,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 2,
        "gamma": 0
    },
    {
        "max_depth": 5,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 2,
        "gamma": 0
    },
    {
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 2,
        "gamma": 0.1
    },
    {
        "max_depth": 7,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 2,
        "gamma": 0.1
    },
    {
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.5,
        "reg_lambda": 3,
        "gamma": 0.1
    },
    {
        "max_depth": 5,
        "min_child_weight": 1,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1,
        "gamma": 0
    },
    {
        "max_depth": 6,
        "min_child_weight": 3,
        "subsample": 0.9,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.5,
        "reg_lambda": 3,
        "gamma": 0.2
    },
    {
        "max_depth": 5,
        "min_child_weight": 7,
        "subsample": 0.7,
        "colsample_bytree": 0.8,
        "reg_alpha": 1,
        "reg_lambda": 5,
        "gamma": 0.2
    }
]

In [26]:
tuning_results = []
tuned_models = []

for i, params in enumerate(param_configs, start=1):
    print(f"\nTraining configuration {i}/{len(param_configs)}")

    candidate = XGBClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        early_stopping_rounds=75,
        n_jobs=4,
        random_state=42,
        verbosity=0,
        **params
    )

    candidate.fit(
        X_train_final,
        y_train,
        eval_set=[(X_valid_final, y_valid)],
        verbose=False
    )

    probabilities = candidate.predict_proba(
        X_valid_final
    )[:, 1]

    score = log_loss(y_valid, probabilities)

    tuning_results.append({
        "Configuration": i,
        "Log Loss": score,
        "Best Iteration": candidate.best_iteration,
        **params
    })

    tuned_models.append(candidate)

    print(
        f"Log Loss: {score:.6f} | "
        f"Best iteration: {candidate.best_iteration}"
    )

tuning_df = pd.DataFrame(tuning_results).sort_values(
    "Log Loss"
).reset_index(drop=True)

print("\nTuning results:")
print(tuning_df.to_string(index=False))


Training configuration 1/8
Log Loss: 0.457361 | Best iteration: 1499

Training configuration 2/8
Log Loss: 0.453953 | Best iteration: 1499

Training configuration 3/8
Log Loss: 0.451613 | Best iteration: 1499

Training configuration 4/8
Log Loss: 0.450064 | Best iteration: 1484

Training configuration 5/8
Log Loss: 0.457663 | Best iteration: 1499

Training configuration 6/8
Log Loss: 0.453937 | Best iteration: 1499

Training configuration 7/8
Log Loss: 0.451880 | Best iteration: 1499

Training configuration 8/8
Log Loss: 0.454209 | Best iteration: 1499

Tuning results:
 Configuration  Log Loss  Best Iteration  max_depth  min_child_weight  subsample  colsample_bytree  reg_alpha  reg_lambda  gamma
             4  0.450064            1484          7                 5        0.8               0.8        0.1           2    0.1
             3  0.451613            1499          6                 5        0.8               0.8        0.1           2    0.1
             7  0.451880            

In [27]:
best_config_index = int(
    tuning_df.loc[0, "Configuration"]
) - 1

best_xgb_model = tuned_models[best_config_index]

best_prob = best_xgb_model.predict_proba(
    X_valid_final
)[:, 1]

best_pred = (best_prob >= 0.5).astype("int8")

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

tuned_metrics = {
    "Log Loss": log_loss(y_valid, best_prob),
    "Accuracy": accuracy_score(y_valid, best_pred),
    "Precision": precision_score(y_valid, best_pred),
    "Recall": recall_score(y_valid, best_pred),
    "F1-score": f1_score(y_valid, best_pred)
}

print("Best tuned XGBoost results")
for name, value in tuned_metrics.items():
    print(f"{name}: {value:.6f}")

Best tuned XGBoost results
Log Loss: 0.450064
Accuracy: 0.755114
Precision: 0.655700
Recall: 0.709007
F1-score: 0.681313


In [28]:
comparison_df = pd.DataFrame([
    {
        "Model": "Original XGBoost",
        "Log Loss": 0.451301,
        "Accuracy": 0.754100,
        "F1-score": 0.680538
    },
    {
        "Model": "Tuned XGBoost",
        "Log Loss": tuned_metrics["Log Loss"],
        "Accuracy": tuned_metrics["Accuracy"],
        "F1-score": tuned_metrics["F1-score"]
    }
])

print(comparison_df.to_string(index=False))

           Model  Log Loss  Accuracy  F1-score
Original XGBoost  0.451301  0.754100  0.680538
   Tuned XGBoost  0.450064  0.755114  0.681313


In [29]:
import joblib
from pathlib import Path

MODEL_DIR = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\models"
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the tuned XGBoost model
joblib.dump(
    best_xgb_model,
    MODEL_DIR / "xgboost_tuned.joblib"
)

# Save the fitted TF-IDF vectorizer
joblib.dump(
    word_vectorizer,
    MODEL_DIR / "word_tfidf_vectorizer.joblib"
)

# Save feature names and model configuration
feature_metadata = {
    "feature_names": X_train_final.columns.tolist(),
    "best_iteration": int(best_xgb_model.best_iteration),
    "best_score": float(best_xgb_model.best_score)
}

joblib.dump(
    feature_metadata,
    MODEL_DIR / "feature_metadata.joblib"
)

print("Tuned model and supporting artifacts saved successfully.")

Tuned model and supporting artifacts saved successfully.


In [31]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

SUBMISSION_PATH = Path(
    r"C:\Users\bimal\OneDrive\Desktop\satya\ML Projects"
    r"\question_similarity\data\processed\submission.csv"
)

BATCH_SIZE = 5000

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)

# Validate required columns
assert "test_id" in test_df.columns
assert {"question1", "question2"}.issubset(test_df.columns)

# Generate predictions in batches
for start in tqdm(
    range(0, len(test_df), BATCH_SIZE),
    desc="Generating predictions"
):
    end = min(start + BATCH_SIZE, len(test_df))
    batch = test_df.iloc[start:end]

    # Handle missing values and ensure text columns contain strings
    batch_questions = batch[
        ["question1", "question2"]
    ].fillna("").astype(str)

    # Generate handcrafted features
    hand_features = create_handcrafted_features(batch_questions)

    # Transform questions using the fitted TF-IDF vectorizer
    q1_matrix = word_vectorizer.transform(
        batch_questions["question1"]
    )
    q2_matrix = word_vectorizer.transform(
        batch_questions["question2"]
    )

    # Calculate cosine similarity
    word_similarity = np.asarray(
        q1_matrix.multiply(q2_matrix).sum(axis=1)
    ).ravel().astype(np.float32)

    # Combine features in the exact training order
    batch_features = hand_features.copy()
    batch_features["word_tfidf_similarity"] = word_similarity
    batch_features = batch_features[X_train_final.columns]

    # Generate duplicate probabilities
    probabilities = best_xgb_model.predict_proba(
        batch_features
    )[:, 1]

    # Create the submission batch
    submission_batch = pd.DataFrame({
        "test_id": batch["test_id"].values,
        "is_duplicate": probabilities
    })

    # Write each batch directly to disk
    submission_batch.to_csv(
        SUBMISSION_PATH,
        mode="w" if start == 0 else "a",
        header=(start == 0),
        index=False,
        float_format="%.8f"
    )

print(f"\nSubmission saved to: {SUBMISSION_PATH}")

Generating predictions: 100%|██████████| 713/713 [03:23<00:00,  3.51it/s]


Submission saved to: C:\Users\bimal\OneDrive\Desktop\satya\ML Projects\question_similarity\data\processed\submission.csv


In [32]:
submission_df = pd.read_csv(SUBMISSION_PATH)

print("Submission shape:", submission_df.shape)
print("\nFirst five rows:")
print(submission_df.head())

assert len(submission_df) == len(test_df)
assert submission_df.columns.tolist() == [
    "test_id", "is_duplicate"
]
assert submission_df["is_duplicate"].notna().all()
assert submission_df["is_duplicate"].between(0, 1).all()
assert submission_df["test_id"].notna().all()

print("\nSubmission validation passed.")

Submission shape: (3563475, 2)

First five rows:
  test_id  is_duplicate
0       0      0.011188
1       1      0.600039
2       2      0.548595
3       3      0.000299
4       4      0.345518

Submission validation passed.


C:\Users\bimal\AppData\Local\Temp\ipykernel_4228\1167017760.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  submission_df = pd.read_csv(SUBMISSION_PATH)


In [33]:
submission_df = pd.read_csv(
    SUBMISSION_PATH,
    dtype={"test_id": str},
    low_memory=False
)

print("Submission shape:", submission_df.shape)
print(submission_df.head())

Submission shape: (3563475, 2)
  test_id  is_duplicate
0       0      0.011188
1       1      0.600039
2       2      0.548595
3       3      0.000299
4       4      0.345518
